# 🎵 Music to MIDI — Google Colab (Free GPU)

在 Colab 免费 T4 GPU 上运行 Music to MIDI，通过 Gradio 公开链接访问。

**使用方法**：
1. 点击上方菜单 **Runtime → Change runtime type → T4 GPU**
2. 依次运行下面的单元格
3. 最后一个单元格会输出一个公开链接，打开即可使用

In [ ]:
#@title 1️⃣ 检查 GPU 可用性（详细日志）
import subprocess
import sys
from datetime import datetime

import torch


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


log("Starting runtime diagnostics")
log(f"Python version: {sys.version.split()[0]}")
log(f"PyTorch version: {torch.__version__}")
log(f"Torch CUDA build: {torch.version.cuda}")
log(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)
    total_vram_gb = props.total_memory / 1024**3
    log(f"Detected GPU: {props.name}")
    log(f"Total VRAM: {total_vram_gb:.2f} GB")
    log(f"SM count: {props.multi_processor_count}")
    try:
        reserved = torch.cuda.memory_reserved(device_index) / 1024**3
        allocated = torch.cuda.memory_allocated(device_index) / 1024**3
        log(f"Allocated VRAM: {allocated:.3f} GB")
        log(f"Reserved VRAM: {reserved:.3f} GB")
    except Exception as exc:
        log(f"Failed to read VRAM stats: {exc}")

    log("nvidia-smi summary:")
    try:
        smi = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,driver_version",
                "--format=csv,noheader",
            ],
            text=True,
        ).strip()
        print(smi, flush=True)
    except Exception as exc:
        log(f"Unable to run nvidia-smi: {exc}")

    log("GPU runtime check passed")
else:
    log("No GPU detected. In Colab use Runtime -> Change runtime type -> T4 GPU")
    raise RuntimeError("GPU runtime is required")


In [ ]:
#@title 2️⃣ 克隆仓库并安装依赖（详细日志）
import importlib
import os
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def run_cmd(command, cwd=None):
    log(f"CMD: {command}")
    start = time.time()
    proc = subprocess.Popen(
        command,
        cwd=cwd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    line_count = 0
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
        line_count += 1
    return_code = proc.wait()
    elapsed = time.time() - start
    log(f"command finished: exit={return_code}, lines={line_count}, elapsed={elapsed:.1f}s")
    if return_code != 0:
        raise RuntimeError(f"command failed: {command}")


repo_dir = Path("/content/music-to-midi")
repo_url = "https://github.com/mason369/music-to-midi.git"

log("准备项目工作区")
if not repo_dir.exists():
    run_cmd(f"git clone {repo_url} {repo_dir}")
else:
    log("仓库已存在，显示状态并同步远程")
    run_cmd("git -C /content/music-to-midi remote -v")
    run_cmd("git -C /content/music-to-midi rev-parse --short HEAD")
    run_cmd("git -C /content/music-to-midi fetch --all --prune")
    run_cmd("git -C /content/music-to-midi status -sb")

os.chdir(repo_dir)
log(f"工作目录: {os.getcwd()}")

log("升级 pip/setuptools/wheel")
run_cmd("python -m pip install --upgrade pip setuptools wheel")

# ── 记录 Colab 预装的 torch 版本信息（不要动它） ──
log("检测 Colab 预装 torch 版本")
import torch
torch_base = torch.__version__.split("+")[0]
cuda_tag = torch.__version__.split("+")[1] if "+" in torch.__version__ else ""
log(f"torch=={torch.__version__} (base={torch_base}, cuda_tag={cuda_tag})")
log(f"CUDA available: {torch.cuda.is_available()}, CUDA version: {torch.version.cuda}")

# torch 与 torchvision/torchaudio 严格对应关系
version_map = {
    "2.10.0": ("0.25.0", "2.10.0"),
    "2.5.1":  ("0.20.1", "2.5.1"),
    "2.5.0":  ("0.20.0", "2.5.0"),
    "2.4.1":  ("0.19.1", "2.4.1"),
    "2.4.0":  ("0.19.0", "2.4.0"),
    "2.3.1":  ("0.18.1", "2.3.1"),
    "2.3.0":  ("0.18.0", "2.3.0"),
}
tv_ver, ta_ver = version_map.get(torch_base, ("0.20.1", "2.5.1"))
log(f"对应版本: torchvision=={tv_ver}, torchaudio=={ta_ver}")

# 确定 pip index（使用 Colab 预装的 CUDA 标签）
if cuda_tag:
    torch_index = f"https://download.pytorch.org/whl/{cuda_tag}"
elif torch.cuda.is_available():
    cuda_version = (torch.version.cuda or "").replace(".", "")
    torch_index = f"https://download.pytorch.org/whl/cu{cuda_version}"
else:
    torch_index = "https://download.pytorch.org/whl/cpu"
log(f"PyTorch index: {torch_index}")

log("安装 Python 依赖（完整输出模式）")
packages = [
    "lightning",
    "einops",
    "transformers==4.45.1",
    "mir-eval",
    "deprecated",
    "smart-open",
    "nest-asyncio",
    "scipy",
    "scikit-learn",
    "torchmetrics",
    "wandb",
    "huggingface_hub",
    "mido",
    "librosa",
    "soundfile",
    "audio-separator>=0.36.0",
    "aria-amt@git+https://github.com/EleutherAI/aria-amt.git",
    "onnxruntime",
    "gradio>=4.44.0",
    "tqdm",
    "pyyaml",
    "psutil",
    "numba",
]
quoted_packages = " ".join(shlex.quote(pkg) for pkg in packages)
run_cmd("python -m pip install " + quoted_packages)

# ── 修复 aria-amt 降级 torch 系列的问题 ──
# aria-amt 要求 torchaudio<=2.5，会把 torch 降级到 2.5.0 并安装不兼容的 torchvision
# 用 --no-deps 只恢复这三个包的版本，不拉入新的 CUDA 底层库，保留 Colab 系统 CUDA
log("修复依赖安装后的 torch 系列版本（--no-deps 保留系统 CUDA 库）")
run_cmd(
    "python -m pip install --no-deps --force-reinstall "
    f"torch=={torch.__version__} torchaudio=={ta_ver}+{cuda_tag} torchvision=={tv_ver}+{cuda_tag} "
    f"--index-url {torch_index}"
)

log("验证 torch/torchaudio/torchvision 兼容性")
# 强制重新加载模块以验证
for mod_name in ["torch", "torchaudio", "torchvision"]:
    if mod_name in __import__("sys").modules:
        del __import__("sys").modules[mod_name]
try:
    import torch as _t
    import torchaudio as _ta
    import torchvision as _tv
    log(f"torch=={_t.__version__}")
    log(f"torchaudio=={_ta.__version__}")
    log(f"torchvision=={_tv.__version__}")
except Exception as exc:
    log(f"警告: 验证失败 ({exc})，请在安装完成后 Restart runtime 再继续")

log("使用 apt 安装 FFmpeg（完整输出）")
run_cmd("apt-get update")
run_cmd("apt-get install -y ffmpeg")

log("关键包版本")
for module_name in ["torch", "torchaudio", "torchvision", "gradio", "huggingface_hub", "lightning", "librosa", "amt"]:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        log(f"{module_name}=={version}")
    except Exception as exc:
        log(f"无法读取 {module_name} 版本: {exc}")

log("依赖安装完成（建议 Restart runtime 后再运行后续步骤）")

In [ ]:
#@title 3️⃣ 下载 YourMT3+ 源码和模型权重（详细日志）
import os
import sys
import time
from datetime import datetime
from pathlib import Path

from huggingface_hub import list_repo_files, snapshot_download

os.chdir("/content/music-to-midi")


def log(message=""):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}", flush=True)


def format_size(num_bytes):
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024 or unit == "GB":
            if unit == "B":
                return f"{int(size)} {unit}"
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} GB"


def collect_local_file_info(base_dir, relative_paths):
    file_info = {}
    base_path = Path(base_dir)
    for relative_path in relative_paths:
        local_path = base_path / relative_path
        if local_path.exists():
            file_info[relative_path] = local_path.stat().st_size
    return file_info


repo_id = "mimbres/YourMT3"
yourmt3_dir = "/content/music-to-midi/space/YourMT3"
amt_src = os.path.join(yourmt3_dir, "amt", "src")

log("准备 YourMT3 源码同步（详细模式）")
log(f"目标目录: {yourmt3_dir}")
log(
    "HF_TOKEN 状态: 已设置"
    if os.environ.get("HF_TOKEN")
    else "HF_TOKEN 状态: 未设置（公开仓库可匿名下载）"
)

source_start = time.time()
repo_source_files = sorted(
    file_path
    for file_path in list_repo_files(repo_id, repo_type="space")
    if file_path.startswith("amt/src/")
)

if not repo_source_files:
    raise RuntimeError("远程仓库中未找到 amt/src 文件，请重试。")

log(f"远程源文件数量: {len(repo_source_files)}")
source_before = collect_local_file_info(yourmt3_dir, repo_source_files)
log(f"同步前本地缓存文件数: {len(source_before)}")
log("源文件计划（同步前）：")
for index, file_path in enumerate(repo_source_files, 1):
    if file_path in source_before:
        log(f"  [{index:03d}/{len(repo_source_files)}] CACHE {file_path} ({format_size(source_before[file_path])})")
    else:
        log(f"  [{index:03d}/{len(repo_source_files)}] FETCH {file_path}")

snapshot_download(
    repo_id,
    repo_type="space",
    local_dir=yourmt3_dir,
    allow_patterns=["amt/src/**"],
    ignore_patterns=["*.ckpt", "*.bin", "*.safetensors", "amt/logs/**"],
)

source_after = collect_local_file_info(yourmt3_dir, repo_source_files)
new_files = [file_path for file_path in repo_source_files if file_path not in source_before and file_path in source_after]
cached_files = [file_path for file_path in repo_source_files if file_path in source_before and file_path in source_after]

log("源文件结果（同步后）：")
for index, file_path in enumerate(repo_source_files, 1):
    if file_path in source_after:
        status = "NEW" if file_path in new_files else "CACHE"
        log(f"  [{index:03d}/{len(repo_source_files)}] {status:5} {file_path} ({format_size(source_after[file_path])})")
    else:
        log(f"  [{index:03d}/{len(repo_source_files)}] MISSING {file_path}")

source_total_size = sum(source_after.values())
source_elapsed = time.time() - source_start
log(
    f"YourMT3 源码就绪: 新增={len(new_files)}, 缓存={len(cached_files)}, "
    f"总计={format_size(source_total_size)}, 耗时={source_elapsed:.1f}秒"
)

# Create symlink: project_root/YourMT3 -> space/YourMT3
root_link = "/content/music-to-midi/YourMT3"
if not os.path.exists(root_link):
    os.symlink(yourmt3_dir, root_link)
    log("符号链接已创建: YourMT3 -> space/YourMT3")
else:
    log("符号链接已存在: YourMT3 -> space/YourMT3")

# Add amt/src to sys.path
if amt_src not in sys.path:
    sys.path.insert(0, amt_src)
    log("已将 amt/src 添加到 sys.path")

# Download model weights
log("开始检查/下载模型权重（首次运行约 800 MB）")
sys.path.insert(0, "/content/music-to-midi")
from download_sota_models import download_ultimate_moe

model_cache_root = Path(os.path.expanduser("~/.cache/music_ai_models/yourmt3_all"))
existing_ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
if existing_ckpts:
    log("现有检查点缓存：")
    for idx, ckpt in enumerate(existing_ckpts, 1):
        log(f"  [{idx:02d}/{len(existing_ckpts)}] {ckpt} ({format_size(ckpt.stat().st_size)})")
else:
    log("本地未找到检查点缓存；预计首次下载")

download_ultimate_moe()

final_ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
if final_ckpts:
    log("下载后的检查点文件：")
    for idx, ckpt in enumerate(final_ckpts, 1):
        log(f"  [{idx:02d}/{len(final_ckpts)}] {ckpt} ({format_size(ckpt.stat().st_size)})")
else:
    log("警告：下载后未检测到 .ckpt 文件，请检查上方日志。")

log("模型权重就绪")


In [ ]:
#@title 4️⃣ 启动 Gradio 应用（详细日志 + 公开链接）
import logging
import os
import platform
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path


def tlog(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


os.chdir("/content/music-to-midi")

# ── 杀掉旧 Gradio 进程（避免端口占用）──
try:
    result = subprocess.run(
        ["fuser", "-k", "7860/tcp"],
        capture_output=True,
        text=True,
        timeout=5,
    )
    if result.stdout.strip() or result.stderr.strip():
        tlog("Port 7860 cleanup output:")
        if result.stdout.strip():
            print(result.stdout.strip(), flush=True)
        if result.stderr.strip():
            print(result.stderr.strip(), flush=True)
except Exception as exc:
    tlog(f"Skip port cleanup: {exc}")

# ── 路径设置 ──
PROJECT_ROOT = "/content/music-to-midi"
amt_src = "/content/music-to-midi/space/YourMT3/amt/src"
for p in [PROJECT_ROOT, amt_src]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── 环境变量 ──
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"
os.environ["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

# ── 日志 ──
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
logger = logging.getLogger("colab-app")

import gradio as gr
import torch


# ── 启动前诊断信息 ──
logger.info("========== Startup Diagnostics ==========")
logger.info(f"Python version: {platform.python_version()}")
logger.info(f"Gradio version: {gr.__version__}")
logger.info(f"Torch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"YourMT3 source path: {amt_src}")
logger.info(f"YourMT3 ymt3.py exists: {Path(amt_src, 'model', 'ymt3.py').exists()}")

# ── 验证 YourMT3 可导入 ──
try:
    from model.ymt3 import YourMT3  # noqa: F401
    logger.info("✅ YourMT3 code import success")
except ImportError as e:
    raise RuntimeError(f"YourMT3 import failed: {e}, 请重新运行第 3 步")

# ── 导入核心模块 ──
from src.models.data_models import Config, ProcessingStage
from src.core.pipeline import MusicToMidiPipeline
from src.utils.gpu_utils import clear_gpu_memory

# ── 设备信息 ──
device_label = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
logger.info(f"Current device: {device_label}")

# ── 模型缓存检查 ──
model_cache_root = Path(os.path.expanduser("~/.cache/music_ai_models/yourmt3_all"))
ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
logger.info(f"Model cache root: {model_cache_root}")
logger.info(f"Checkpoint count: {len(ckpts)}")
for idx, ckpt in enumerate(ckpts, 1):
    size_mb = ckpt.stat().st_size / (1024 * 1024)
    logger.info(f"  [{idx:02d}/{len(ckpts)}] {ckpt} ({size_mb:.1f} MB)")

# ── 日志文件（供前端轮询）──
LOG_FILE = "/tmp/midi_process.log"


class _RobustFileHandler(logging.Handler):
    def __init__(self, filename):
        super().__init__()
        self.filename = filename

    def emit(self, record):
        try:
            msg = self.format(record)
            with open(self.filename, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception:
            pass


_fh = _RobustFileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S"))
_fh.setLevel(logging.INFO)
for _name in ("colab-app", "src.core", "src.utils"):
    logging.getLogger(_name).addHandler(_fh)


def clear_logs():
    try:
        open(LOG_FILE, "w").close()
    except Exception:
        pass


def read_logs():
    try:
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
            content = f.read()
        content = content.replace("\x00", "")
        lines = content.strip().split("\n")
        return "\n".join(lines[-200:]) if lines and lines[0] else ""
    except Exception as e:
        return f"[error] {e}"


# ── 核心转换函数 ──
def convert_audio_to_midi(
    audio_path,
    mode,
    quality,
    vocal_split_merge_midi=False,
    six_stem_only_selected=False,
    six_stem_targets=None,
    six_stem_vocal_harmony=False,
    progress=gr.Progress(),
):
    if audio_path is None:
        raise gr.Error("请先上传音频文件")

    clear_logs()
    logger.info("========== New Conversion ==========")
    logger.info(f"Audio file: {Path(audio_path).name}")
    logger.info(f"Mode: {mode}")
    logger.info(f"Quality: {quality}")

    config = Config()
    mode_mapping = {
        "YourMT3+ 多乐器转写": "smart",
        "人声分离 + 分别转写": "vocal_split",
        "六声部分离 + 分别转写": "six_stem_split",
        "Piano Dedicated (Aria-AMT)": "piano_aria_amt",
    }
    config.processing_mode = mode_mapping.get(mode, "smart")
    config.transcription_quality = quality
    config.vocal_split_merge_midi = bool(
        config.processing_mode == "vocal_split" and vocal_split_merge_midi
    )

    if config.processing_mode == "six_stem_split" and six_stem_only_selected:
        config.six_stem_targets = [str(stem).lower() for stem in (six_stem_targets or [])]
        logger.info(f"Six-stem selective mode: {config.six_stem_targets}")
    else:
        config.six_stem_targets = []
        if config.processing_mode == "six_stem_split":
            logger.info("Six-stem full mode: transcribe all 6 stems")
    config.six_stem_split_vocal_harmony = bool(
        config.processing_mode == "six_stem_split" and six_stem_vocal_harmony
    )

    pipeline = MusicToMidiPipeline(config)
    output_dir = tempfile.mkdtemp(prefix="midi_output_")
    logger.info(f"Output dir: {output_dir}")

    def on_progress(p):
        stage_name = {
            ProcessingStage.PREPROCESSING: "预处理",
            ProcessingStage.SEPARATION: "音源分离",
            ProcessingStage.TRANSCRIPTION: "音频转写",
            ProcessingStage.VOCAL_TRANSCRIPTION: "人声转写",
            ProcessingStage.SYNTHESIS: "MIDI合成",
            ProcessingStage.COMPLETE: "完成",
        }.get(p.stage, str(p.stage))
        logger.info(f"Progress {p.overall_progress * 100:5.1f}% [{stage_name}] {p.message}")
        progress(p.overall_progress, desc=f"[{stage_name}] {p.message}")

    try:
        result = pipeline.process(audio_path=audio_path, output_dir=output_dir, progress_callback=on_progress)
    except Exception as e:
        logger.error(f"Conversion failed: {e}")
        raise gr.Error(f"转换失败: {e}")
    finally:
        try:
            clear_gpu_memory()
            logger.info("GPU memory cleaned")
        except Exception as e:
            logger.warning(f"GPU memory cleanup warning: {e}")

    output_files = []
    if result.midi_path and Path(result.midi_path).exists():
        output_files.append(result.midi_path)
    if result.stem_midi_paths:
        for stem_name, stem_midi_path in result.stem_midi_paths.items():
            if stem_midi_path and Path(stem_midi_path).exists() and stem_midi_path not in output_files:
                output_files.append(stem_midi_path)
    if result.vocal_midi_path and Path(result.vocal_midi_path).exists():
        output_files.append(result.vocal_midi_path)
    if result.accompaniment_midi_path and Path(result.accompaniment_midi_path).exists():
        if result.accompaniment_midi_path != result.midi_path:
            output_files.append(result.accompaniment_midi_path)

    # 分离后的音频文件（人声 + 伴奏 WAV 或六声部 stem WAV）
    if result.separated_audio:
        for audio_key, audio_file in result.separated_audio.items():
            if audio_file and Path(audio_file).exists():
                output_files.append(audio_file)

    # 去重并输出文件清单
    dedup_files = []
    seen = set()
    for file_path in output_files:
        if file_path not in seen:
            seen.add(file_path)
            dedup_files.append(file_path)
    output_files = dedup_files

    logger.info(f"Output file count: {len(output_files)}")
    for idx, file_path in enumerate(output_files, 1):
        p = Path(file_path)
        size_mb = p.stat().st_size / (1024 * 1024)
        logger.info(f"  [{idx:02d}/{len(output_files)}] {p.name} ({size_mb:.2f} MB)")

    bpm_str = f"{result.beat_info.bpm:.1f}" if result.beat_info else "N/A"
    status_lines = [
        "--- 转换完成 ---",
        f"耗时: {result.processing_time:.1f} 秒",
        f"总音符数: {result.total_notes}",
        f"BPM: {bpm_str}",
        f"设备: {device_label}",
        f"输出文件数: {len(output_files)}",
    ]
    if result.stem_midi_paths:
        status_lines.append(f"合并 MIDI: {Path(result.midi_path).name}")
        status_lines.append(f"分 stem MIDI: {len(result.stem_midi_paths)} 个")
    elif result.vocal_midi_path:
        status_lines.append(f"伴奏 MIDI: {Path(result.accompaniment_midi_path).name}")
        status_lines.append(f"人声 MIDI: {Path(result.vocal_midi_path).name}")
        if result.merged_midi_path:
            status_lines.append(f"合并 MIDI: {Path(result.merged_midi_path).name}")
    else:
        status_lines.append(f"MIDI 文件: {Path(result.midi_path).name}")

    logger.info("========== Conversion Finished ==========")
    return output_files, "\n".join(status_lines)


def update_mode_info(mode):
    if mode == "人声分离 + 分别转写":
        return "**BS-RoFormer + YourMT3+** — 先分离人声与伴奏，再分别转写；默认输出两个独立 MIDI，可选额外输出 1 个合并 MIDI。"
    if mode == "六声部分离 + 分别转写":
        return "**BS Roformer SW + YourMT3+** — 分离 bass/drums/guitar/piano/vocals/other 六个 stem，默认转写 6 个 stem（可切换为仅转写选中 stem），并额外生成 1 个合并 MIDI。"
    if mode == "Piano Dedicated (Aria-AMT)":
        return "**Aria-AMT (Piano)** — dedicated piano-only transcription model for solo piano input."
    return "**YourMT3+ MoE** — 直接对完整音频进行多乐器转写，精确识别 128 种 GM 乐器。"


def update_mode_controls(mode, only_selected):
    is_vocal = mode == "人声分离 + 分别转写"
    is_six = mode == "六声部分离 + 分别转写"
    return (
        update_mode_info(mode),
        gr.update(visible=is_vocal),
        gr.update(visible=is_six),
        gr.update(visible=(is_six and bool(only_selected))),
        gr.update(visible=is_six),
    )


def update_six_stem_targets_visibility(mode, only_selected):
    return gr.update(visible=(mode == "六声部分离 + 分别转写" and bool(only_selected)))


# ── JavaScript 日志轮询（通过 head 注入，兼容 share 链接）──
LOG_POLL_JS = """<script>
(function() {
    var pollCount = 0;
    var _pollTimer = setInterval(function() {
        pollCount++;
        var ta = document.querySelector('.log-box textarea');
        if (!ta) return;
        var setter = Object.getOwnPropertyDescriptor(
            HTMLTextAreaElement.prototype, 'value'
        ).set;
        fetch('./api/read_logs', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({data: []})
        })
        .then(function(r) { return r.json(); })
        .then(function(json) {
            var logText = (json.data && json.data[0]) ? json.data[0] : '';
            if (logText) {
                setter.call(ta, logText);
            }
            ta.dispatchEvent(new Event('input', {bubbles: true}));
            ta.scrollTop = ta.scrollHeight;
        })
        .catch(function() {});
    }, 1200);
})();
</script>"""


# ── Gradio 界面 ──
with gr.Blocks(
    title="Music to MIDI (Colab GPU)",
    head=LOG_POLL_JS,
    theme=gr.themes.Base(primary_hue=gr.themes.colors.blue, neutral_hue=gr.themes.colors.slate),
) as demo:
    gr.Markdown("# 🎵 Music to MIDI\n在 Google Colab 免费 GPU 上运行 — 基于 YourMT3+ MoE 深度学习模型")

    with gr.Row():
        with gr.Column(scale=5):
            audio_input = gr.Audio(label="上传音频文件", type="filepath")
            gr.Markdown("<small>支持 MP3, WAV, FLAC, OGG, M4A</small>")
            mode_radio = gr.Radio(
                ["YourMT3+ 多乐器转写", "人声分离 + 分别转写", "六声部分离 + 分别转写", "Piano Dedicated (Aria-AMT)"],
                value="YourMT3+ 多乐器转写",
                label="处理模式",
            )
            mode_info = gr.Markdown("**YourMT3+ MoE** — 直接对完整音频进行多乐器转写，精确识别 128 种 GM 乐器。")
            vocal_split_merge_midi = gr.Checkbox(
                value=False,
                label="输出 1 个人声+伴奏合并 MIDI（人声分离模式）",
                visible=False,
            )
            six_stem_only_selected = gr.Checkbox(
                value=False,
                label="仅转写选中的 stem（六声部模式）",
                visible=False,
            )
            six_stem_targets = gr.CheckboxGroup(
                choices=["bass", "drums", "guitar", "piano", "vocals", "other"],
                value=["bass", "drums", "guitar", "piano", "vocals", "other"],
                label="六声部转写目标",
                info="仅在勾选'仅转写选中的 stem'且模式为六声部分离时生效",
                visible=False,
            )
            six_stem_vocal_harmony = gr.Checkbox(
                value=False,
                label="Split vocals into lead + harmony (experimental)",
                visible=False,
            )
            mode_radio.change(
                fn=update_mode_controls,
                inputs=[mode_radio, six_stem_only_selected],
                outputs=[mode_info, vocal_split_merge_midi, six_stem_only_selected, six_stem_targets, six_stem_vocal_harmony],
                api_name=False,
            )
            six_stem_only_selected.change(
                fn=update_six_stem_targets_visibility,
                inputs=[mode_radio, six_stem_only_selected],
                outputs=[six_stem_targets],
                api_name=False,
            )
            quality_radio = gr.Radio(["fast", "balanced", "best"], value="balanced", label="转写质量")
            convert_btn = gr.Button("▶ 开始转换", variant="primary", size="lg")
            gr.Markdown(f"当前设备: **{device_label}**")

        with gr.Column(scale=5):
            status_output = gr.Textbox(label="状态", interactive=False, lines=7, placeholder="等待转换...")
            file_output = gr.File(label="下载文件", file_count="multiple")
            log_output = gr.Textbox(label="控制台日志", interactive=False, lines=16, max_lines=30, elem_classes="log-box")

    convert_btn.click(
        fn=convert_audio_to_midi,
        inputs=[audio_input, mode_radio, quality_radio, vocal_split_merge_midi, six_stem_only_selected, six_stem_targets, six_stem_vocal_harmony],
        outputs=[file_output, status_output],
    )

    # 日志 API 端点（供 JavaScript 轮询调用）
    _log_poll_btn = gr.Button(visible=False)
    _log_poll_btn.click(
        fn=read_logs,
        inputs=[],
        outputs=[log_output],
        api_name="read_logs",
        queue=False,
    )

    gr.Markdown("<center>基于 [YourMT3+](https://github.com/mimbres/YourMT3) MoE 模型 | [GitHub](https://github.com/mason369/music-to-midi)</center>")

print("\n" + "=" * 60)
print("🚀 启动中... 公开链接将在下方显示（详细日志模式）")
print("=" * 60 + "\n")
demo.launch(share=True, server_name="0.0.0.0", server_port=7860)